# **REINVENT Transfer Learning and Sampling Workflow**

This notebook performs the target-specific transfer learning and sampling workflow for the REINVENT generative model. It sets up the REINVENT4 environment, creates the required folder structure, and splits the prepared SMILES dataset into training and validation subsets. Using a pretrained prior model, the notebook writes the necessary transfer learning and sampling configuration files, runs transfer learning for the selected target, and generates molecules from both the final trained model and intermediate checkpoints. The outputs, including checkpoints, sampled molecules, logs, and configuration files, are saved in an organized target-specific directory for downstream validation and analysis.


Notes:

- Keep running REINVENT via conda run -n reinvent4 ….

- Run RDKit code (validation/plots) in the notebook kernel, where pip install rdkit worked

- Adjust target once at the top.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# One-time: Miniforge
!pip -q install condacolab
!python - <<'PY'
import condacolab; condacolab.install_miniforge()

/bin/bash: line 1: warning: here-document at line 1 delimited by end-of-file (wanted `PY')
⏬ Downloading https://github.com/jaimergp/miniforge/releases/download/24.11.2-1_colab/Miniforge3-colab-24.11.2-1_colab-Linux-x86_64.sh...
📦 Installing...
📌 Adjusting configuration...
🩹 Patching environment...
⏲ Done in 0:00:11
🔁 Restarting kernel...


In [ ]:
# sanity: repo exists + has pyproject
!test -d /content/REINVENT4 || git clone https://github.com/MolecularAI/REINVENT4.git /content/REINVENT4
!test -f /content/REINVENT4/pyproject.toml && echo "OK: pyproject found" || echo "pyproject MISSING"

# create env (safe to re-run) + upgrade pip
!conda create -n reinvent4 python=3.10 -y
!conda run -n reinvent4 python -m pip install -U pip

# >>> IMPORTANT: run installer *from the repo dir* <<<
!conda run -n reinvent4 bash -lc 'cd /content/REINVENT4 && python install.py cpu'

# checks (use module fallback if the CLI script isn’t on PATH)
!conda run -n reinvent4 bash -lc 'reinvent --version || python -m reinvent.cli --version'
!conda run -n reinvent4 bash -lc 'reinvent --help    | head -n 20 || python -m reinvent.cli --help | head -n 20'


Cloning into '/content/REINVENT4'...
remote: Enumerating objects: 3110, done.
remote: Counting objects: 100% (577/577), done.
remote: Compressing objects: 100% (154/154), done.
remote: Total 3110 (delta 459), reused 434 (delta 423), pack-reused 2533 (from 3)
Receiving objects: 100% (3110/3110), 1.25 GiB | 28.40 MiB/s, done.
Resolving deltas: 100% (1617/1617), done.
OK: pyproject found
Channels:
 - conda-forge
Platform: linux-64
Solving environment: \ | done


==> WARNING: A newer version of conda exists. <==
    current version: 24.11.2
    latest version: 25.7.0

Please update conda by running

    $ conda update -n base -c conda-forge conda



## Package Plan ##

  environment location: /usr/local/envs/reinvent4

  added / updated specs:
    - python=3.10


The following packages will be downloaded:

    package                    |            build
    ---------------------------|-----------------
    bzip2-1.0.8                |       hda65f42_8         254 KB  conda-forge
    

In [ ]:
from pathlib import Path
import csv, math, statistics as stats


import pandas as pd


In [ ]:
# =========================
# 0) Imports & target setup
# =========================
from pathlib import Path
from sklearn.model_selection import train_test_split

# --- choose target here ---
targets = ["5-HT6", "ache", "bace1", "buche", "esr1", "3beta", "mao-b"]
target  = targets[6]   # e.g., "5-HT6"

# --- paths ---
workdir    = Path("/content/drive/MyDrive/Colab Notebooks/ESOFT-MSC/thesis-project/GenAI/reinvent")
data_root  = Path("/content/drive/MyDrive/Colab Notebooks/ESOFT-MSC/thesis-project/GenAI/data/training-data")
smi_source = data_root / f"{target}_smiles_for_training.smi"   # your current single SMI list

# per-target working tree
TROOT       = workdir / target
CKPTS_DIR   = TROOT / "ckpts";    CKPTS_DIR.mkdir(parents=True, exist_ok=True)
SAMPLES_DIR = TROOT / "samples";  SAMPLES_DIR.mkdir(parents=True, exist_ok=True)
LOGS_DIR    = TROOT / "logs";     LOGS_DIR.mkdir(parents=True, exist_ok=True)
CFG_DIR     = TROOT / "configs";  CFG_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR    = TROOT / "data";     DATA_DIR.mkdir(parents=True, exist_ok=True)

print("Using source SMI:", smi_source)


Using source SMI: /content/drive/MyDrive/Colab Notebooks/ESOFT-MSC/thesis-project/GenAI/data/training-data/mao-b_smiles_for_training.smi


In [ ]:
# 1) Train/Val split from your existing SMI
from sklearn.model_selection import train_test_split

# read + basic clean
smiles = [l.strip() for l in smi_source.read_text(encoding="utf-8").splitlines() if l.strip()]

# 15% validation; fixed seed for reproducibility
train_smi, val_smi = train_test_split(smiles, test_size=0.15, random_state=123, shuffle=True)

# write split files into the target's data folder
(TRAIN_PATH := DATA_DIR / "train.smi").write_text("\n".join(train_smi), encoding="utf-8")
(VAL_PATH   := DATA_DIR / "val.smi").write_text("\n".join(val_smi),   encoding="utf-8")

print(f"[Split] Train: {len(train_smi)}   Val: {len(val_smi)}")
print("train.smi ->", TRAIN_PATH)
print("val.smi   ->", VAL_PATH)


[Split] Train: 2513   Val: 444
train.smi -> /content/drive/MyDrive/Colab Notebooks/ESOFT-MSC/thesis-project/GenAI/reinvent/mao-b/data/train.smi
val.smi   -> /content/drive/MyDrive/Colab Notebooks/ESOFT-MSC/thesis-project/GenAI/reinvent/mao-b/data/val.smi


# **2) Ensure prior exists, write TL + base sampling config**

In [ ]:
# 2) Make sure the classical prior is present (skip if already downloaded)
PRIOR_DIR  = Path("/content/REINVENT4/priors"); PRIOR_DIR.mkdir(parents=True, exist_ok=True)
prior_path = PRIOR_DIR / "reinvent.prior"
if not prior_path.exists():
    import subprocess
    url = "https://zenodo.org/records/15641297/files/reinvent.prior?download=1"
    print("[DL] downloading classical prior …")
    subprocess.run(["wget", "-q", "-O", str(prior_path), url], check=True)
print("[OK] Prior at:", prior_path.as_posix())

# 2a) TL config — trains ONLY on data/train.smi and saves checkpoints every 2 epochs
save_model = CKPTS_DIR / "TL_agent.model"
tl_cfg     = CFG_DIR / "transfer_learning.toml"
tl_cfg.write_text(f"""
run_type = "transfer_learning"
tb_logdir = "logs/_tb_TL"
json_out_config = "_tl_dump.json"

[parameters]
input_model_file  = "{prior_path.as_posix()}"
output_model_file = "{save_model.as_posix()}"
smiles_file       = "{(DATA_DIR/'train.smi').as_posix()}"

num_epochs = 12
batch_size = 32
save_every_n_epochs = 2
""".strip()+"\n", encoding="utf-8")
print("[OK] wrote TL config →", tl_cfg)

# 2b) a base sampling config for the final model (you’ll also get per-epoch configs later)
samp_cfg_final = CFG_DIR / "sampling_final.toml"
samp_csv_final = SAMPLES_DIR / "sampling_final.csv"
samp_cfg_final.write_text(f"""
run_type = "sampling"
json_out_config = "_sampling_dump_final.json"

[parameters]
model_file       = "{save_model.as_posix()}"
num_smiles       = 2000
unique_molecules = true
randomize_smiles = true
output_file      = "{samp_csv_final.as_posix()}"
""".strip()+"\n", encoding="utf-8")
print("[OK] wrote sampling (final) config →", samp_cfg_final)


[DL] downloading classical prior …
[OK] Prior at: /content/REINVENT4/priors/reinvent.prior
[OK] wrote TL config → /content/drive/MyDrive/Colab Notebooks/ESOFT-MSC/thesis-project/GenAI/reinvent/mao-b/configs/transfer_learning.toml
[OK] wrote sampling (final) config → /content/drive/MyDrive/Colab Notebooks/ESOFT-MSC/thesis-project/GenAI/reinvent/mao-b/configs/sampling_final.toml


# **3) Run TL (in reinvent4 env) and sample the final model**

In [ ]:
# 3) Train + sample final model inside the conda env
import subprocess

# Train (CPU here; switch -d cuda if you installed the CUDA build)
cmd_tl = f'cd "{TROOT.as_posix()}" && reinvent -d cpu -s 123 -l "logs/transfer.log" "configs/{tl_cfg.name}"'
print("[Run] TL:", cmd_tl)
subprocess.run(["bash", "-lc", f"conda run -n reinvent4 bash -lc '{cmd_tl}'"], check=False)

# Sample final
cmd_sp_final = f'cd "{TROOT.as_posix()}" && reinvent -d cpu -s 123 -l "logs/sampling_final.log" "configs/{samp_cfg_final.name}"'
print("[Run] Sampling (final):", cmd_sp_final)
subprocess.run(["bash", "-lc", f"conda run -n reinvent4 bash -lc '{cmd_sp_final}'"], check=False)

print("[OK] sampling_final exists?", samp_csv_final.exists(), "→", samp_csv_final)


[Run] TL: cd "/content/drive/MyDrive/Colab Notebooks/ESOFT-MSC/thesis-project/GenAI/reinvent/mao-b" && reinvent -d cpu -s 123 -l "logs/transfer.log" "configs/transfer_learning.toml"
[Run] Sampling (final): cd "/content/drive/MyDrive/Colab Notebooks/ESOFT-MSC/thesis-project/GenAI/reinvent/mao-b" && reinvent -d cpu -s 123 -l "logs/sampling_final.log" "configs/sampling_final.toml"
[OK] sampling_final exists? True → /content/drive/MyDrive/Colab Notebooks/ESOFT-MSC/thesis-project/GenAI/reinvent/mao-b/samples/sampling_final.csv


# **4) Sample every checkpoint → sampling_epoch_XX.csv**

In [ ]:
import re, subprocess
from pathlib import Path

# ---- robust epoch parser: supports `.model.2.chkpt`, `_E02.model`, `epoch_06.*` ----
def epoch_from_name(p: Path):
    name = p.name
    # TL_agent.model.2.chkpt  -> 2
    m = re.search(r"\.model\.(\d+)\.chkpt$", name)
    if m: return int(m.group(1))
    # TL_agent_E02.model      -> 2
    m = re.search(r"[_\-]E(\d+)", name)
    if m: return int(m.group(1))
    # sampling_epoch_06.csv   -> 6 (fallback if you ever reuse this)
    m = re.search(r"epoch[_\-]?(\d+)", name)
    if m: return int(m.group(1))
    return None

# ---- collect checkpoints (both .chkpt and .model) ----
candidates = []
for pat in ("*.chkpt", "*.model", "TL_agent.model.*.chkpt"):
    candidates += list(CKPTS_DIR.glob(pat))

# de-dup and keep only files with an epoch number OR the final model
seen = set()
epoched = []
for p in candidates:
    if not p.is_file():
        continue
    key = (p.name, p.stat().st_size)
    if key in seen:
        continue
    seen.add(key)
    ep = epoch_from_name(p)
    if ep is not None:
        epoched.append((p, ep))

epoched.sort(key=lambda x: x[1])

final_model = CKPTS_DIR / "TL_agent.model"   # from your TL config
has_final   = final_model.exists()

print("Found checkpoints:")
for p, ep in epoched:
    print(f"  E{ep:02d}  {p.name}")
print("Final model present?", has_final, "→", final_model.name)

# ---- write a sampling config and run reinvent for each checkpoint ----
def write_sampling_cfg(model_path: Path, out_csv: Path, tag: str):
    cfg_path = CFG_DIR / f"sampling_{tag}.toml"
    cfg_path.write_text(f"""
run_type = "sampling"
json_out_config = "_sampling_dump_{tag}.json"

[parameters]
model_file       = "{model_path.as_posix()}"
num_smiles       = 2000
unique_molecules = true
randomize_smiles = true
output_file      = "{out_csv.as_posix()}"
""".strip()+"\n", encoding="utf-8")
    return cfg_path

# sample each epoch checkpoint
for mdl, ep in epoched:
    tag = f"epoch_{ep:02d}"
    out_csv = SAMPLES_DIR / f"sampling_{tag}.csv"
    cfg     = write_sampling_cfg(mdl, out_csv, tag)
    cmd     = f'cd "{TROOT.as_posix()}" && reinvent -d cpu -s 123 -l "logs/sampling_{tag}.log" "configs/{cfg.name}"'
    print(f"[Run] Sampling {mdl.name} → {out_csv.name}")
    subprocess.run(["bash", "-lc", f"conda run -n reinvent4 bash -lc '{cmd}'"], check=False)

# also (re)sample the final model to sampling_final.csv
if has_final:
    out_csv = SAMPLES_DIR / "sampling_final.csv"
    cfg     = write_sampling_cfg(final_model, out_csv, "final")
    cmd     = f'cd "{TROOT.as_posix()}" && reinvent -d cpu -s 123 -l "logs/sampling_final.log" "configs/{cfg.name}"'
    print(f"[Run] Sampling final → {out_csv.name}")
    subprocess.run(["bash", "-lc", f"conda run -n reinvent4 bash -lc '{cmd}'"], check=False)

print("\n[Done] Sampling files under:", SAMPLES_DIR.as_posix())
for f in sorted(SAMPLES_DIR.glob("sampling*.csv")):
    print(" ", f.name, f.stat().st_size, "bytes")


Found checkpoints:
  E02  TL_agent.model.2.chkpt
  E04  TL_agent.model.4.chkpt
  E06  TL_agent.model.6.chkpt
  E08  TL_agent.model.8.chkpt
  E10  TL_agent.model.10.chkpt
  E12  TL_agent.model.12.chkpt
Final model present? True → TL_agent.model
[Run] Sampling TL_agent.model.2.chkpt → sampling_epoch_02.csv
[Run] Sampling TL_agent.model.4.chkpt → sampling_epoch_04.csv
[Run] Sampling TL_agent.model.6.chkpt → sampling_epoch_06.csv
[Run] Sampling TL_agent.model.8.chkpt → sampling_epoch_08.csv
[Run] Sampling TL_agent.model.10.chkpt → sampling_epoch_10.csv
[Run] Sampling TL_agent.model.12.chkpt → sampling_epoch_12.csv
[Run] Sampling final → sampling_final.csv

[Done] Sampling files under: /content/drive/MyDrive/Colab Notebooks/ESOFT-MSC/thesis-project/GenAI/reinvent/mao-b/samples
  sampling_epoch_02.csv 97935 bytes
  sampling_epoch_04.csv 94989 bytes
  sampling_epoch_06.csv 93933 bytes
  sampling_epoch_08.csv 93474 bytes
  sampling_epoch_10.csv 91770 bytes
  sampling_epoch_12.csv 90748 bytes
 

In [ ]:
# 4) Create per-epoch sampling CSVs for ALL checkpoints saved during TL
import re, subprocess

def epoch_num(p: Path):
    m = re.search(r"_E(\d+)\.model$", p.name)
    return int(m.group(1)) if m else None

# collect checkpoints like TL_agent_E02.model, TL_agent_E04.model, ...
ckpt_models = sorted([p for p in CKPTS_DIR.glob("*.model") if epoch_num(p) is not None],
                     key=lambda p: epoch_num(p))

if not ckpt_models:
    print("[Warn] No explicit epoch checkpoints found in", CKPTS_DIR)

for mdl in ckpt_models:
    ep = epoch_num(mdl)
    tag = f"epoch_{ep:02d}"
    out_csv = SAMPLES_DIR / f"sampling_{tag}.csv"
    tmp_cfg = CFG_DIR / f"sampling_{tag}.toml"

    tmp_cfg.write_text(f"""
run_type = "sampling"
json_out_config = "_sampling_dump_{tag}.json"

[parameters]
model_file       = "{mdl.as_posix()}"
num_smiles       = 2000
unique_molecules = true
randomize_smiles = true
output_file      = "{out_csv.as_posix()}"
""".strip()+"\n", encoding="utf-8")

    cmd = f'cd "{TROOT.as_posix()}" && reinvent -d cpu -s 123 -l "logs/sampling_{tag}.log" "configs/{tmp_cfg.name}"'
    print(f"[Run] Sampling {mdl.name} → {out_csv.name}")
    subprocess.run(["bash", "-lc", f"conda run -n reinvent4 bash -lc '{cmd}'"], check=False)

print("\n[Done] Per-epoch sampling files are in:", SAMPLES_DIR.as_posix())


[Warn] No explicit epoch checkpoints found in /content/drive/MyDrive/Colab Notebooks/ESOFT-MSC/thesis-project/GenAI/reinvent/mao-b/ckpts

[Done] Per-epoch sampling files are in: /content/drive/MyDrive/Colab Notebooks/ESOFT-MSC/thesis-project/GenAI/reinvent/mao-b/samples
